# MinCED CRISPR Array Detection Example

This notebook demonstrates how to detect CRISPR arrays in nucleotide sequences using **MinCED** (Mining CRISPRs in Environmental Datasets).

MinCED identifies CRISPR repeat-spacer arrays in genomic DNA, reporting repeat consensus sequences, individual spacers, and array coordinates.

## Imports

In [1]:
from bio_programming_tools import MincedConfig, MincedInput, run_minced


## 1. Detect CRISPR Arrays

Find CRISPR arrays in a DNA sequence. MinCED scans for repeated sequences interspersed with unique spacers — the hallmark of CRISPR loci.

### API Reference

**`MincedInput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Nucleotide sequences to scan for CRISPR arrays |
| `sequence_ids` | `Optional[List[str]]` | Identifiers for each sequence (auto-generated if omitted) |

**`MincedConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `min_num_repeats` | `int` | `3` | Minimum number of repeats to call a CRISPR array |
| `min_repeat_length` | `int` | `27` | Minimum repeat length in nucleotides |

**`MincedOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `results` | `List[MincedSequenceResult]` | Per-sequence CRISPR detection results |
| `metadata` | `Dict` | Execution metadata |

In [2]:
# Synthetic CRISPR array: leader + 5 repeat-spacer units + final repeat + trailer
# Each repeat is 36nt, each spacer is 31nt — mimics a real bacterial CRISPR locus
REPEAT = "GTTTTAGAGCTATGCTGTTTTGAATGGTCCCAAAAC"   # 36nt repeat
SPACER_1 = "AAAAAAAACCCCCCCCTTTTTTTTGGGGGGGG"       # 31nt spacer
SPACER_2 = "CCCCCCCCAAAAAAAGGGGGGGTTTTTTTTAA"       # 31nt spacer
SPACER_3 = "TTTTTTTTGGGGGGGGAAAAAAAACCCCCCCC"       # 31nt spacer
SPACER_4 = "GGGGGGGGTTTTTTTTCCCCCCCCAAAAAAAA"       # 31nt spacer
SPACER_5 = "AACCCCGGTTTTAAAACCCCGGGGTTTTAAAA"       # 31nt spacer

crispr_sequence = (
    "ATCGATCGATCGATCGATCGATCGATCG"  # Leader
    + REPEAT + SPACER_1
    + REPEAT + SPACER_2
    + REPEAT + SPACER_3
    + REPEAT + SPACER_4
    + REPEAT + SPACER_5
    + REPEAT                         # Final repeat (no spacer)
    + "ATCGATCGATCGATCGATCGATCGATCG"  # Trailer
)

inputs = MincedInput(
    sequences=[crispr_sequence],
    sequence_ids=["test_genome"],
)

config = MincedConfig(
    min_num_repeats=3,
    min_repeat_length=27,
)

result = run_minced(inputs, config)

for seq_result in result.results:
    print(f"{seq_result.sequence_id}: {seq_result.num_arrays} CRISPR arrays found")
    if seq_result.has_crispr:
        for i, array in enumerate(seq_result.crispr_arrays):
            print(f"  Array {i+1}: {array.num_repeats} repeats")
            print(f"  Spacers: {array.spacers[:3]}..." if len(array.spacers) > 3 else f"  Spacers: {array.spacers}")

test_genome: 1 CRISPR arrays found
  Array 1: 6 repeats
  Spacers: ['AAAAAAAACCCCCCCCTTTTTTTTGGGGGGGG', 'CCCCCCCCAAAAAAAGGGGGGGTTTTTTTTAA', 'TTTTTTTTGGGGGGGGAAAAAAAACCCCCCCC']...


## 2. Export Results

Save CRISPR detection results in CSV or JSON format.

In [3]:
from pathlib import Path

output_dir = Path("./example_output")
result.export(name="minced_results", export_path=output_dir, file_format="json")
print(f"CRISPR detection results exported to {output_dir}")

CRISPR detection results exported to example_output
